In [1]:
import cv2
import numpy as np
import os
from pathlib import Path
from tqdm import tqdm

In [2]:
def illumination_correction(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)

    lab = cv2.merge((l, a, b))
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

In [3]:
def denoise(img):
    return cv2.fastNlMeansDenoisingColored(
        img, None,
        h=6, hColor=6,
        templateWindowSize=7,
        searchWindowSize=21
    )

In [4]:
def blur_score(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()

In [5]:
def resize_if_needed(img, min_height=900):
    h, w = img.shape[:2]
    if h < min_height:
        scale = min_height / h
        img = cv2.resize(
            img,
            None,
            fx=scale,
            fy=scale,
            interpolation=cv2.INTER_CUBIC  # text-safe
        )
    return img

In [6]:
def adaptive_sharpen(img):
    score = blur_score(img)

    if score < 80:
        alpha = 1.8
    elif score < 150:
        alpha = 1.4
    else:
        return img  # already sharp enough

    blurred = cv2.GaussianBlur(img, (0, 0), sigmaX=1.0)
    return cv2.addWeighted(img, alpha, blurred, -(alpha - 1), 0)


In [7]:
def preprocess_for_ocr(img):
    img = illumination_correction(img)
    img = denoise(img)
    img = resize_if_needed(img)
    img = adaptive_sharpen(img)

    return img


In [9]:
INPUT_DIR = r"D:\Y4 Research\datasets\dietary Images\Set1"
OUTPUT_DIR = r"D:\Y4 Research\datasets\dietary Images\OCR_ready"

os.makedirs(OUTPUT_DIR, exist_ok=True)

for img_path in tqdm(list(Path(INPUT_DIR).glob("*.*"))):
    img = cv2.imread(str(img_path))
    if img is None:
        continue

    processed = preprocess_for_ocr(img)

    out_path = Path(OUTPUT_DIR) / img_path.name
    cv2.imwrite(str(out_path), processed)


100%|██████████| 100/100 [00:49<00:00,  2.01it/s]


In [10]:
import pytesseract

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

In [11]:
!where tesseract    # Windows

C:\Program Files\Tesseract-OCR\tesseract.exe


INFO: Could not find "#".
INFO: Could not find "Windows".


In [12]:
print(pytesseract.get_tesseract_version())

5.5.0.20241111


In [13]:
import os

In [14]:
INPUT_DIR = r"D:\Y4 Research\datasets\dietary Images\OCR_ready"
OUT_DIR = r"D:\Y4 Research\datasets\dietary Images\OCR"

In [16]:
import json

In [17]:
os.makedirs(OUT_DIR, exist_ok=True)

# -----------------------------
# Tesseract config
# -----------------------------
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
custom_config = r"--oem 3 --psm 6"

# -----------------------------
# OCR loop
# -----------------------------
for img_name in tqdm(os.listdir(INPUT_DIR)):
    if not img_name.lower().endswith((".png", ".jpg", ".jpeg")):
        continue

    img_path = os.path.join(INPUT_DIR, img_name)
    image = cv2.imread(img_path)

    if image is None:
        print("Failed to read:", img_path)
        continue

    # Full OCR text
    ocr_text = pytesseract.image_to_string(image, config=custom_config)

    # OCR with bounding boxes
    data = pytesseract.image_to_data(image, config=custom_config, output_type=pytesseract.Output.DICT)

    words = []
    n = len(data["text"])
    for i in range(n):
        text = data["text"][i].strip()
        if text == "":
            continue
        try:
            conf = int(float(data["conf"][i]))
        except:
            conf = 0
        word_info = {
            "text": text,
            "confidence": conf,
            "bbox": [
                int(data["left"][i]),
                int(data["top"][i]),
                int(data["width"][i]),
                int(data["height"][i])
            ]
        }
        words.append(word_info)

    output_json = {
        "image_id": img_name,
        "ocr_text": ocr_text,
        "words": words
    }

    out_path = os.path.join(OUT_DIR, os.path.splitext(img_name)[0] + ".json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(output_json, f, indent=2, ensure_ascii=False)

100%|██████████| 100/100 [03:56<00:00,  2.37s/it]
